In [1]:
pip install gitpython pandas javalang requests

In [2]:
import requests
import pandas as pd
import time
import re
from urllib.parse import urlparse
from datetime import datetime

# ========================
# CONFIGURATION
# =========================
repo_url = "https://github.com/kestra-io/kestra"
since_date = "2023-05-01T00:00:00Z"
MAX_RESULTS = 4500   # LIMIT HERE

headers = {
    "Accept": "application/vnd.github.v3+json",
    "Authorization": ""   # put your token here
}

keywords = [

# Scheduling tactic (extended)
"scheduling",
"scheduler",
"task scheduling",
"job scheduling",
"process scheduling",
"thread scheduling",
"priority scheduling",
"priority queue",
"task queue",
"job queue",
"work queue",
"queueing",
"fair scheduling",
"round robin",
"round-robin",
"fifo scheduling",
"deadline scheduling",
"rate monotonic",
"earliest deadline first",
"timer",
"timed task",
"periodic task",
"delayed task",
"task dispatcher",
"dispatcher",
"orchestration",
"workflow engine",
"workflow scheduling",
"resource scheduling",
"cpu scheduling",
"load shedding",
"admission control",

"resource allocation",
"resource arbitration",

# Core performance concepts
"slowness",
"bottleneck",
"overhead",

# Resource utilization
"cpu usage",
"memory usage",
"memory leak",
"garbage collection",
"resource consumption",

# Optimization tactics
"optimize",
"optimization",
"efficient",
"efficiency",
"speed up",

# Caching tactics
"cache",
"caching",
"memoization",

# Concurrency & parallelism
"thread",
"multithreading",
"concurrency",
"parallel",
"asynchronous",
"async",
"non-blocking",
"blocking",

# Load handling
"scalability",
"scalable",
"load balancing",
"high load",
"traffic spike",

# I/O performance
"bandwidth",
"serialization",
"deserialization",

# Database performance
"query optimization",
"slow query",

# Architecture-level tactics
"lazy loading",
"batch processing",
"pooling",
"connection pool",
"thread pool",
"buffering",
"streaming",
"pagination",

# Monitoring / detection

"monitoring",
"tracing",

# Fault-related performance issues
"timeout",
"retry",
"backoff",
"throttle",
"rate limit",

# Event-based / heartbeat (extended)
"heartbeat",
"healthcheck",
"health check",
"liveness",
"liveness probe",
"readiness",
"readiness probe",
"keepalive",
"keep-alive",
"ping",
"pong",
"watchdog",
"watchdog timer",
"failure detection",
"failure detector",
"node failure",
"node health",
"cluster health",
"service health",
"crash detection",
"replica health",
"replication lag",
"leader election",

"gossip protocol",
"cluster membership",
"service discovery",
"service registry",
"high availability",
"failover",
"auto recovery",
"self-healing"

]



# =========================
# UTILITIES
# =========================

def extract_repo_info(url):
    path = urlparse(url).path.strip("/")
    parts = path.split("/")
    return parts[0], parts[1]

def contains_keyword(text, keywords):
    if not text:
        return False
    text = text.lower()
    return any(keyword in text for keyword in keywords)

def clean_text(text):
    if isinstance(text, str):
        return re.sub(r'[\x00-\x1F\x7F]', '', text)
    return text

def handle_rate_limit(response):
    if response.status_code == 403:
        reset_time = int(response.headers.get("X-RateLimit-Reset", 0))
        remaining = int(response.headers.get("X-RateLimit-Remaining", 1))

        if remaining == 0:
            sleep_seconds = max(reset_time - int(time.time()), 0) + 5
            reset_datetime = datetime.fromtimestamp(reset_time)
            print(f"⏳ Rate limit exceeded. Sleeping until {reset_datetime}...")
            time.sleep(sleep_seconds)
            return True
    return False

# =========================
# MAIN MINING
# =========================

owner, repo = extract_repo_info(repo_url)
print(f"🔍 Mining repository: {owner}/{repo}")

session = requests.Session()
session.headers.update(headers)

all_commits = []
page = 1
scanned_commits = 0

while len(all_commits) < MAX_RESULTS:

    api_url = f"https://api.github.com/repos/{owner}/{repo}/commits"

    params = {
        "since": since_date,
        "per_page": 100,
        "page": page
    }

    response = session.get(api_url, params=params)

    if handle_rate_limit(response):
        continue

    if response.status_code != 200:
        print(f"❌ Error: {response.status_code}")
        break

    commits = response.json()

    if not commits:
        print("No more commits.")
        break

    print(f"Processing page {page} ({len(commits)} commits)...")

    for commit in commits:

        scanned_commits += 1
        message = commit["commit"]["message"]

        if contains_keyword(message, keywords):

            commit_time = commit["commit"]["author"]["date"]

            author_name = (
                commit["author"]["login"]
                if commit.get("author")
                else commit["commit"]["author"]["name"]
            )

            all_commits.append({
                "Repository": f"{owner}/{repo}",
                "Commit ID": commit["sha"],
                "URL": commit["html_url"],
                "Description": message,
                "Commit Time": commit_time,
                "Author": author_name
            })

            # 🔴 Stop immediately when reaching 4500
            if len(all_commits) >= MAX_RESULTS:
                print(f"Reached {MAX_RESULTS} matching commits. Stopping.")
                break

    page += 1
    time.sleep(0.4)  # polite delay

print(f"\nScanned commits: {scanned_commits}")
print(f"Matched commits: {len(all_commits)}")

# =========================
# SAVE RESULTS
# =========================

df = pd.DataFrame(all_commits)

df = df.applymap(clean_text)

if not df.empty:
    excel_filename = f"Java_Commits_{repo}_filtered.xlsx"
    df.to_excel(excel_filename, index=False)
    print(f"Saved Excel: {excel_filename}")
else:
    print("No matching commits found.")


🔍 Mining repository: kestra-io/kestra
📄 Page 1 | Remaining API calls: 4996
Processing page 1 (100 commits)
📄 Page 2 | Remaining API calls: 4995
Processing page 2 (100 commits)
📄 Page 3 | Remaining API calls: 4994
Processing page 3 (100 commits)
📄 Page 4 | Remaining API calls: 4993
Processing page 4 (100 commits)
📄 Page 5 | Remaining API calls: 4992
Processing page 5 (100 commits)
📄 Page 6 | Remaining API calls: 4991
Processing page 6 (100 commits)
📄 Page 7 | Remaining API calls: 4990
Processing page 7 (100 commits)
📄 Page 8 | Remaining API calls: 4989
Processing page 8 (100 commits)
📄 Page 9 | Remaining API calls: 4988
Processing page 9 (100 commits)
📄 Page 10 | Remaining API calls: 4987
Processing page 10 (100 commits)
📄 Page 11 | Remaining API calls: 4986
Processing page 11 (100 commits)
📄 Page 12 | Remaining API calls: 4985
Processing page 12 (100 commits)
📄 Page 13 | Remaining API calls: 4984
Processing page 13 (100 commits)
📄 Page 14 | Remaining API calls: 4983
Processing page 14 

/tmp/ipykernel_2021/1535227260.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


💾 Saved: Java_Commits_kestra_checkpoint.xlsx
📄 Page 21 | Remaining API calls: 4976
Processing page 21 (100 commits)
📄 Page 22 | Remaining API calls: 4975
Processing page 22 (100 commits)
📄 Page 23 | Remaining API calls: 4974
Processing page 23 (100 commits)
📄 Page 24 | Remaining API calls: 4973
Processing page 24 (100 commits)
📄 Page 25 | Remaining API calls: 4972
Processing page 25 (100 commits)
📄 Page 26 | Remaining API calls: 4971
Processing page 26 (100 commits)
📄 Page 27 | Remaining API calls: 4970
Processing page 27 (100 commits)
📄 Page 28 | Remaining API calls: 4969
Processing page 28 (100 commits)
📄 Page 29 | Remaining API calls: 4968
Processing page 29 (100 commits)
📄 Page 30 | Remaining API calls: 4967
Processing page 30 (100 commits)
📄 Page 31 | Remaining API calls: 4966
Processing page 31 (100 commits)
📄 Page 32 | Remaining API calls: 4965
Processing page 32 (100 commits)
📄 Page 33 | Remaining API calls: 4964
Processing page 33 (100 commits)
📄 Page 34 | Remaining API calls: